# Assignment 2 – PS06: Loan Approval Expert System

**Course:** M.Tech in Artificial Intelligence and Machine Learning  
**Assignment:** Assignment 2 – PS06  
**Group ID:** G166

## Group Contribution Declaration
| Member Name | Student ID | Contribution (%) |
|-------------|-----------|-----------------|
| Hakeem Sharath   | 2025AA05665    | 100%             |
| Harisha   | 2025AA05669    | 100%             |
|  Azhar Ansari  | 2025AB05055    | 100%            |
| ANANDHALAKSHMI| 2025aa05683 | 100% |
| Chanchal | 2025aa05494 | 100% |

## Objective

This notebook is a step-by-step implementation and testing notebook for the
required **Prolog** loan-approval expert system.

The system uses:

1. Annual Income
2. CIBIL Score
3. Previous Loan Defaults
4. Employment History Duration

Final target classes:

- `APPROVED`
- `APPROVED (Lower Credit Limit)`
- `APPROVED (With Co-Signer)`
- `REJECTED`

The notebook creates the single required Prolog source file:
`loan_approval.pl`.

## 1. Decision Tree Interpretation

### High-Income Path

If Annual Income >= ₹5,00,000:

- CIBIL >= 750 → check previous defaults.
  - No default → APPROVED.
  - Previous default + Employment >= 1 year → APPROVED (Lower Credit Limit).
  - Previous default + Employment < 1 year → REJECTED.
- CIBIL 700–749 → check employment.
  - Employment >= 1 year → APPROVED (Lower Credit Limit).
  - Employment < 1 year → REJECTED.
- CIBIL < 700 → REJECTED.

### Low-Income Path

If Annual Income < ₹5,00,000:

- Income >= ₹3,00,000:
  - CIBIL >= 750 → APPROVED (With Co-Signer).
  - CIBIL < 750 → REJECTED.
- Income < ₹3,00,000 → REJECTED.

## 2. Notebook Setup

The actual decision logic will be written in **Prolog**, because the
assignment requires a Prolog source file.

Python is used only by this notebook as a helper to:

- create `loan_approval.pl`,
- run SWI-Prolog when it is installed,
- create `inputPS166.txt`,
- create `outputPS166.txt`,
- and perform repeatable tests.

The submitted decision-making implementation remains the Prolog file.

In [1]:
from pathlib import Path
import subprocess
import shutil

GROUP_ID = "G166"
PL_FILE = Path("loan_approval.pl")
INPUT_FILE = Path("inputPS166.txt")
OUTPUT_FILE = Path("outputPS166.txt")

SWIPL = shutil.which("swipl")

print("Group ID:", GROUP_ID)
print("Prolog executable:", SWIPL if SWIPL else "Not found")

Group ID: G166
Prolog executable: C:\Program Files\swipl\bin\swipl.EXE


## 3. Prolog Decision Rules

The next code cell contains the complete knowledge base.

The main predicate is:

`loan_decision(Income, CIBIL, Defaults, Employment, Decision)`

The rules are kept modular so that each branch of the Decision Tree is easy
to identify and explain in the design document.

In [2]:
# Write the complete Prolog program into the single required .pl file.

prolog_source = r'''/* ================================================================

   SINGLE Prolog source file required by the assignment.

   Attributes:
     Income      - Annual income
     CIBIL       - CIBIL score
     Defaults    - yes/no previous loan defaults
     Employment  - employment history in years

   ================================================================ */


/* ================================================================
   HIGH-INCOME PATH
   Income >= Rs. 5,00,000
   ================================================================ */


/* CIBIL >= 750 + no previous default -> APPROVED. */
high_income_decision(Income, CIBIL, no, _Employment, approved) :-
    Income >= 500000,
    CIBIL >= 750.


/* CIBIL >= 750 + previous default + employment >= 1 year
   -> APPROVED (Lower Credit Limit). */
high_income_decision(
    Income, CIBIL, yes, Employment, approved_lower_credit_limit
) :-
    Income >= 500000,
    CIBIL >= 750,
    Employment >= 1.


/* CIBIL >= 750 + previous default + employment < 1 year
   -> REJECTED. */
high_income_decision(
    Income, CIBIL, yes, Employment, rejected
) :-
    Income >= 500000,
    CIBIL >= 750,
    Employment < 1.


/* CIBIL 700-749 + employment >= 1 year
   -> APPROVED (Lower Credit Limit). */
high_income_decision(
    Income, CIBIL, _Defaults, Employment, approved_lower_credit_limit
) :-
    Income >= 500000,
    CIBIL >= 700,
    CIBIL < 750,
    Employment >= 1.


/* CIBIL 700-749 + employment < 1 year
   -> REJECTED. */
high_income_decision(
    Income, CIBIL, _Defaults, Employment, rejected
) :-
    Income >= 500000,
    CIBIL >= 700,
    CIBIL < 750,
    Employment < 1.


/* CIBIL < 700 -> REJECTED. */
high_income_decision(
    Income, CIBIL, _Defaults, _Employment, rejected
) :-
    Income >= 500000,
    CIBIL < 700.


/* ================================================================
   LOW-INCOME PATH
   Income < Rs. 5,00,000
   ================================================================ */


/* Income Rs. 3,00,000 to below Rs. 5,00,000
   and CIBIL >= 750 -> APPROVED (With Co-Signer). */
low_income_decision(
    Income, CIBIL, _Defaults, _Employment, approved_with_cosigner
) :-
    Income >= 300000,
    Income < 500000,
    CIBIL >= 750.


/* Income Rs. 3,00,000 to below Rs. 5,00,000
   and CIBIL < 750 -> REJECTED. */
low_income_decision(
    Income, CIBIL, _Defaults, _Employment, rejected
) :-
    Income >= 300000,
    Income < 500000,
    CIBIL < 750.


/* Income below Rs. 3,00,000 -> REJECTED. */
low_income_decision(
    Income, _CIBIL, _Defaults, _Employment, rejected
) :-
    Income < 300000.


/* ================================================================
   MAIN DECISION PREDICATE
   ================================================================ */


/* High-income applicants use the High-Income Path. */
loan_decision(Income, CIBIL, Defaults, Employment, Decision) :-
    Income >= 500000,
    high_income_decision(
        Income, CIBIL, Defaults, Employment, Decision
    ).


/* Low-income applicants use the Low-Income Path. */
loan_decision(Income, CIBIL, Defaults, Employment, Decision) :-
    Income < 500000,
    low_income_decision(
        Income, CIBIL, Defaults, Employment, Decision
    ).


/* ================================================================
   INPUT VALIDATION
   ================================================================ */


/* Annual income must be a non-negative number. */
valid_income(Income) :-
    number(Income),
    Income >= 0.


/* CIBIL must be a non-negative number. */
valid_cibil(CIBIL) :-
    number(CIBIL),
    CIBIL >= 0.


/* Employment duration must be a non-negative number. */
valid_employment(Employment) :-
    number(Employment),
    Employment >= 0.


/* Previous defaults must be yes or no. */
valid_default(no).
valid_default(yes).


/* ================================================================
   USER INPUT
   ================================================================ */


/* Read and validate Annual Income. */
get_income(Income) :-
    repeat,
    write('Enter Annual Income (numeric value): '),
    read(Input),
    (
        valid_income(Input)
        ->
        Income = Input,
        !
        ;
        write('Invalid income. Please enter a non-negative numeric value.'),
        nl,
        fail
    ).


/* Read and validate CIBIL Score. */
get_cibil(CIBIL) :-
    repeat,
    write('Enter CIBIL Score (numeric value): '),
    read(Input),
    (
        valid_cibil(Input)
        ->
        CIBIL = Input,
        !
        ;
        write('Invalid CIBIL score. Please enter a non-negative numeric value.'),
        nl,
        fail
    ).


/* Read and validate previous-default status. */
get_defaults(Defaults) :-
    repeat,
    write('Enter Any Past Defaults? (yes/no): '),
    read(Input),
    (
        valid_default(Input)
        ->
        Defaults = Input,
        !
        ;
        write('Invalid input. Please enter yes or no.'),
        nl,
        fail
    ).


/* Read and validate employment history. */
get_employment(Employment) :-
    repeat,
    write('Enter Employment History Duration in years (numeric value): '),
    read(Input),
    (
        valid_employment(Input)
        ->
        Employment = Input,
        !
        ;
        write('Invalid employment duration. Please enter a non-negative numeric value.'),
        nl,
        fail
    ).


/* ================================================================
   OUTPUT
   ================================================================ */


/* Display the exact human-readable final decision. */
display_decision(approved) :-
    write('Final Decision: APPROVED'),
    nl.

display_decision(approved_lower_credit_limit) :-
    write('Final Decision: APPROVED (Lower Credit Limit)'),
    nl.

display_decision(approved_with_cosigner) :-
    write('Final Decision: APPROVED (With Co-Signer)'),
    nl.

display_decision(rejected) :-
    write('Final Decision: REJECTED'),
    nl.


/* ================================================================
   MAIN USER INTERFACE
   ================================================================ */


/* Main predicate requested in the assignment. */
evaluate_loan :-
    nl,
    write('--- Bank Loan Approval Expert System ---'),
    nl,

    get_income(Income),
    get_cibil(CIBIL),
    get_defaults(Defaults),
    get_employment(Employment),

    loan_decision(
        Income, CIBIL, Defaults, Employment, Decision
    ),

    display_decision(Decision).
'''

PL_FILE.write_text(prolog_source, encoding="utf-8")

print("Created:", PL_FILE.resolve())
print("Lines:", len(prolog_source.splitlines()))

Created: D:\GitHub\BITS.Mtech\SEM-2\ACI\loan_approval.pl
Lines: 274


## 4. Verify the Prolog Knowledge Base

The following tests call `loan_decision/5` directly.

This lets us verify the Decision Tree before testing the interactive
`evaluate_loan/0` predicate.

In [3]:
def run_prolog_goal(goal):
    if SWIPL is None:
        return "SWI-Prolog is not installed. Run the generated .pl file in SWI-Prolog."

    result = subprocess.run(
        [SWIPL, "-q", "-s", str(PL_FILE), "-g", goal, "-t", "halt"],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        return "ERROR:\n" + result.stderr

    return result.stdout.strip()


mandatory_tests = [
    ("Scenario 1",
     "loan_decision(600000,700,no,1,D),write(D),nl"),

    ("Scenario 2",
     "loan_decision(495000,780,no,3,D),write(D),nl"),

    ("Scenario 3",
     "loan_decision(1500000,760,yes,5,D),write(D),nl"),
]

for name, goal in mandatory_tests:
    print(name + ":", run_prolog_goal(goal))

Scenario 1: approved_lower_credit_limit
Scenario 2: approved_with_cosigner
Scenario 3: approved_lower_credit_limit


## 5. Mandatory Edge-Case Results

The assignment asks us to report these three scenarios.

| Scenario | Expected Result |
|---|---|
| ₹6,00,000, CIBIL 700, Employment 1, No Defaults | APPROVED (Lower Credit Limit) |
| ₹4,95,000, CIBIL 780, Employment 3, No Defaults | APPROVED (With Co-Signer) |
| ₹15,00,000, CIBIL 760, Employment 5, 1 Previous Default | APPROVED (Lower Credit Limit) |

These are tests only. The actual evaluator may use different values, so
the decision rules do not hard-code these examples.

## 6. Boundary Testing

We also test values around the important thresholds:

- Income ₹5,00,000
- Income ₹3,00,000
- CIBIL 750
- CIBIL 700
- Employment 1 year

This helps catch errors in `<` versus `>=` conditions.

In [4]:
boundary_tests = [
    ("Income 500000, CIBIL 750, no default",
     "loan_decision(500000,750,no,0,D),write(D),nl"),

    ("Income 500000, CIBIL 720, employment 1",
     "loan_decision(500000,720,no,1,D),write(D),nl"),

    ("Income 500000, CIBIL 720, employment 0",
     "loan_decision(500000,720,no,0,D),write(D),nl"),

    ("Income 300000, CIBIL 750",
     "loan_decision(300000,750,no,5,D),write(D),nl"),

    ("Income 300000, CIBIL 749",
     "loan_decision(300000,749,no,5,D),write(D),nl"),

    ("Income 299999, CIBIL 800",
     "loan_decision(299999,800,no,5,D),write(D),nl"),

    ("Income 600000, CIBIL 699",
     "loan_decision(600000,699,no,5,D),write(D),nl"),

    ("Income 600000, CIBIL 750, default yes, employment 0",
     "loan_decision(600000,750,yes,0,D),write(D),nl"),

    ("Income 600000, CIBIL 750, default yes, employment 1",
     "loan_decision(600000,750,yes,1,D),write(D),nl"),
]

for description, goal in boundary_tests:
    print(f"{description}: {run_prolog_goal(goal)}")

Income 500000, CIBIL 750, no default: approved
Income 500000, CIBIL 720, employment 1: approved_lower_credit_limit
Income 500000, CIBIL 720, employment 0: rejected
Income 300000, CIBIL 750: approved_with_cosigner
Income 300000, CIBIL 749: rejected
Income 299999, CIBIL 800: rejected
Income 600000, CIBIL 699: rejected
Income 600000, CIBIL 750, default yes, employment 0: rejected
Income 600000, CIBIL 750, default yes, employment 1: approved_lower_credit_limit


## 7. Interactive `evaluate_loan/0`

The assignment requires the user-facing predicate:

```text
?- evaluate_loan.
```

The next cell demonstrates it with the assignment's first sample-style
input.

The notebook supplies the values automatically so that the notebook can
run without waiting for manual keyboard input.

In [5]:
def run_evaluate_loan(income, cibil, defaults, employment):
    if SWIPL is None:
        return "SWI-Prolog is not installed. Run ?- evaluate_loan. manually in SWI-Prolog."

    stdin_data = (
        f"{income}.\n"
        f"{cibil}.\n"
        f"{defaults}.\n"
        f"{employment}.\n"
    )

    result = subprocess.run(
        [SWIPL, "-q", "-s", str(PL_FILE), "-g", "evaluate_loan", "-t", "halt"],
        input=stdin_data,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        return "ERROR:\n" + result.stderr

    return result.stdout


print(run_evaluate_loan(900000, 780, "no", 2))


--- Bank Loan Approval Expert System ---
Enter Annual Income (numeric value): Enter CIBIL Score (numeric value): Enter Any Past Defaults? (yes/no): Enter Employment History Duration in years (numeric value): Final Decision: APPROVED



## 8. Create the Required Input File

The assignment requires `inputPSXX.txt`.

For Group 166 the filename is:

`inputPS166.txt`

The file below contains the three mandatory edge-case scenarios.

In [6]:
input_content = (
    "600000.\n"
    "700.\n"
    "no.\n"
    "1.\n"
    "\n"
    "495000.\n"
    "780.\n"
    "no.\n"
    "3.\n"
    "\n"
    "1500000.\n"
    "760.\n"
    "yes.\n"
    "5.\n"
)

INPUT_FILE.write_text(input_content, encoding="utf-8")

print(INPUT_FILE.read_text(encoding="utf-8"))

600000.
700.
no.
1.

495000.
780.
no.
3.

1500000.
760.
yes.
5.



## 9. Create the Required Output File

The following cell executes the three mandatory cases and saves the
results in:

`outputPS166.txt`

In [10]:
# Execute the three mandatory test cases and save their results.

mandatory_cases = [
    (600000, 700, "no", 1),
    (495000, 780, "no", 3),
    (1500000, 760, "yes", 5),
    (1500000, 700, "no", 0.5)
]

results = []

for income, cibil, defaults, employment in mandatory_cases:
    result = run_evaluate_loan(
        income,
        cibil,
        defaults,
        employment
    )

    results.append(
        f"Income: {income}\n"
        f"CIBIL: {cibil}\n"
        f"Previous Defaults: {defaults}\n"
        f"Employment: {employment} years\n"
        f"{result.strip()}\n"
    )

OUTPUT_FILE.write_text(
    "\n-----------------------------\n".join(results),
    encoding="utf-8"
)

print("outputPS166.txt created successfully.")
print()
print(OUTPUT_FILE.read_text(encoding="utf-8"))

outputPS166.txt created successfully.

Income: 600000
CIBIL: 700
Previous Defaults: no
Employment: 1 years
--- Bank Loan Approval Expert System ---
Enter Annual Income (numeric value): Enter CIBIL Score (numeric value): Enter Any Past Defaults? (yes/no): Enter Employment History Duration in years (numeric value): Final Decision: APPROVED (Lower Credit Limit)

-----------------------------
Income: 495000
CIBIL: 780
Previous Defaults: no
Employment: 3 years
--- Bank Loan Approval Expert System ---
Enter Annual Income (numeric value): Enter CIBIL Score (numeric value): Enter Any Past Defaults? (yes/no): Enter Employment History Duration in years (numeric value): Final Decision: APPROVED (With Co-Signer)

-----------------------------
Income: 1500000
CIBIL: 760
Previous Defaults: yes
Employment: 5 years
--- Bank Loan Approval Expert System ---
Enter Annual Income (numeric value): Enter CIBIL Score (numeric value): Enter Any Past Defaults? (yes/no): Enter Employment History Duration in year

In [8]:
import shutil

swipl = shutil.which("swipl")
print(swipl)

C:\Program Files\swipl\bin\swipl.EXE


## 10. Final Validation Checklist

Before submission:

- [x] Single Prolog source file
- [x] Modular Prolog rules
- [x] Detailed comments in the Prolog code
- [x] All four attributes accepted
- [x] No hard-coded evaluation data in decision rules
- [x] Mandatory scenarios tested
- [x] Boundary cases tested
- [x] `evaluate_loan/0` implemented
- [x] `inputPS166.txt` generated
- [x] `outputPS166.txt` generated

### Remaining submission items

The assignment also requires:

1. `designPSXX_<group id>.pdf`
2. `G166_Contribution.xlsx`
3. `inputPS166.txt`
4. `outputPS166.txt`
5. The single Prolog file
6. Final ZIP named `G166_A2_PS06.zip`

The design document must be no more than four pages and must include an
alternate modeling approach with performance implications.

In [9]:
# Final file check.

for path in [PL_FILE, INPUT_FILE, OUTPUT_FILE]:
    print(("✓" if path.exists() else "✗"), path,
          f"({path.stat().st_size} bytes)" if path.exists() else "")

✓ loan_approval.pl (6795 bytes)
✓ inputPS166.txt (78 bytes)
✓ outputPS166.txt (1048 bytes)
